**Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

**Exploración y Curación de Datos**

*Edición 2026*

----

# Trabajo práctico entregable - parte 2

En esta notebook, vamos a cargar el conjunto de datos de [la compentencia Kaggle](https://www.kaggle.com/dansbecker/melbourne-housing-snapshot) sobre estimación de precios de ventas de propiedades en Melbourne, Australia.

Utilizaremos el conjunto de datos reducido producido por [DanB](https://www.kaggle.com/dansbecker). Hemos subido una copia a un servidor de la Universidad Nacional de Córdoba para facilitar su acceso remoto.

In [28]:
import matplotlib.pyplot as plt
import numpy
import pandas as pd

import seaborn
seaborn.set_context('talk')

from sqlalchemy import create_engine, text

In [29]:
import plotly
plotly.__version__


'5.24.1'

In [30]:
melb_df = pandas.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_df[:3]

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


In [50]:
airbnb_url = 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/airbnb_price_by_zipcode.csv'

df_airbnb = pd.read_csv(airbnb_url)




HTTPError: HTTP Error 404: Not Found

In [31]:
melb_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [32]:
melb_df.dtypes


,0
Suburb,object
Address,object
Rooms,int64
Type,object
Price,float64
Method,object
SellerG,object
Date,object
Distance,float64
Postcode,float64


In [39]:
# Conteo de valores faltantes
melb_df.isna().sum()

,0
Suburb,0
Address,0
Rooms,0
Type,0
Price,0
Method,0
SellerG,0
Date,0
Distance,0
Postcode,0


In [41]:
# Conteo de ceros
cols_cero = melb_df[melb_df == 0].count(axis=0)
cols_cero[cols_cero > 0]

,0
Distance,6
Bedroom2,16
Bathroom,34
Car,1026
Landsize,1939
BuildingArea,17


In [48]:
melb_df.Date.describe()


,Date
count,13580
unique,58
top,27/05/2017
freq,473


In [45]:
melb_df.Postcode.describe()


,Postcode
count,13580.000000
mean,3105.301915
std,90.676964
min,3000.000000
25%,3044.000000
50%,3084.000000
75%,3148.000000
max,3977.000000


## Ejercicio 1 SQL:

1. Crear una base de datos en SQLite utilizando la libreria [SQLalchemy](https://stackoverflow.com/questions/2268050/execute-sql-from-file-in-sqlalchemy).
https://docs.sqlalchemy.org/en/14/core/engines.html#sqlite

2. Ingestar los datos provistos en 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv' en una tabla y el dataset generado en clase con datos de airbnb y sus precios por codigo postal en otra.

3. Validar tipos de columnas antes de guardar. Usá `df.dtypes` para ver los tipos actuales. Prestá especial atención a columnas como `Date` y `Price`: por ejemplo, `Date` puede estar como string en vez de datetime, y `Price` puede venir como string o float. El método `to_sql()` infiere tipos automáticamente, pero puede fallar si los tipos no son los esperados.

4. Implementar consultas en SQL que respondan con la siguiente información:

    - cantidad de registros totales por `Regionname`.
    - cantidad de registros totales por `Suburb` y `Regionname`.
    - Consulta con filtro: ¿Cuántas propiedades hay por `Regionname` con más de 2 habitaciones?
    - Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (`Type`) y `Regionname`?
    - Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio.

5. Combinar los datasets de ambas tablas ingestadas utilizando el comando JOIN de SQL para obtener un resultado similar a lo realizado con Pandas en clase.

6. Agregar una celda de validación posterior al JOIN con assertions o validación de esquema. Como mínimo, verificá que el número de filas no cambió, que no aparecieron nulos inesperados y que los rangos de variables agregadas sean razonables. Esta validación implementa dimensiones básicas de calidad de datos como validez, completitud e integridad.



In [33]:
# Crear el motor de SQLite (archivo local 'melb_housing.db')
engine = create_engine('sqlite:///melb_housing.db', echo=False)

# Ingestar los datos en tablas
melb_df.to_sql('melb_data', con=engine, if_exists='replace', index=False)
#df_airbnb.to_sql('airbnb_data', con=engine, if_exists='replace', index=False)

print("Tablas 'melb_data' y 'airbnb_data' creadas exitosamente.")

Tablas 'melb_data' y 'airbnb_data' creadas exitosamente.


In [37]:
melb_df_2= melb_df.copy()

# 1. Convertir Date a datetime
melb_df_2['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)

# 2. Asegurar que Price sea float
melb_df_2['Price'] = pd.to_numeric(melb_df['Price'], errors='coerce')

# 3. Limpiar Postcode (clave para el JOIN posterior)
melb_df_2['Postcode'] = melb_df['Postcode'].astype(float)
#df_airbnb['zipcode'] = df_airbnb['zipcode'].astype(float)

print("Tipos de datos de Melbourne:\n", melb_df_2.dtypes[['Date', 'Price', 'Postcode']])


Tipos de datos de Melbourne:
 Date        datetime64[ns]
Price              float64
Postcode           float64
dtype: object


In [ ]:
# A. Cantidad de registros totales por Regionname
query_a = "SELECT Regionname, COUNT(*) as total FROM melb_data GROUP BY Regionname"
res_a = pd.read_sql(query_a, engine)

# B. Cantidad de registros totales por Suburb y Regionname
query_b = "SELECT Suburb, Regionname, COUNT(*) as total FROM melb_data GROUP BY Suburb, Regionname"
res_b = pd.read_sql(query_b, engine)

# C. Propiedades por Regionname con más de 2 habitaciones
query_c = """
SELECT Regionname, COUNT(*) as total
FROM melb_data
WHERE Rooms > 2
GROUP BY Regionname
"""
res_c = pd.read_sql(query_c, engine)

# D. Precio promedio según tipo (Type) y Regionname
query_d = """
SELECT Type, Regionname, AVG(Price) as avg_price
FROM melb_data
GROUP BY Type, Regionname
"""
res_d = pd.read_sql(query_d, engine)

# E. Top 5 barrios con propiedades más caras en promedio
query_e = """
SELECT Suburb, AVG(Price) as avg_price
FROM melb_data
GROUP BY Suburb
ORDER BY avg_price DESC
LIMIT 5
"""
res_e = pd.read_sql(query_e, engine)

display(res_e)

## Ejercicio 2 - Pandas:

Este ejercicio usa el archivo `airbnb_price_by_zipcode.csv` generado en el notebook `02.1 Combinación de datasets.ipynb`. Si no lo tenés, generarlo primero antes de comenzar esta parte.

1. Seleccionar un subconjunto de columnas que les parezcan relevantes al problema de predicción del valor de la propiedad. Justificar explicitamente las columnas seleccionadas y las que no lo fueron.
  1. Valores faltantes: ¿Qué porcentaje de filas tienen al menos un valor faltante?
  2. Mostrar la dispersión o distribución de las columnas seleccionadas.
 3. Eliminar los valores extremos que no sean relevantes para la predicción de valores de las propiedades.
 4. Mostrar visualmente los valores extremos que eliminás


2. Agregar información adicional respectiva al entorno de una propiedad a partir del [conjunto de datos de AirBnB](https://www.kaggle.com/tylerx/melbourne-airbnb-open-data?select=cleansed_listings_dec18.csv) utilizado en el práctico.
  1. Seleccionar qué variables agregar y qué combinaciones aplicar a cada una. Por ejemplo, pueden utilizar solo la columna `price`, o aplicar múltiples transformaciones como la mediana (¿por qué no la media?) o el mínimo.
  2. Utilizar la variable zipcode para unir los conjuntos de datos. Sólo incluir los zipcodes que tengan una cantidad mínima de registros (a elección) como para que la información agregada sea relevante.
  3. Mostrar un gráfico zipcode vs airbnb_price_median.
  4. Investigar al menos otras 2 variables que puedan servir para combinar los datos, y justificar si serían adecuadas o no. Pueden asumir que cuentan con la ayuda de anotadores expertos para encontrar equivalencias entre barrios o direcciones, o que cuentan con algoritmos para encontrar las n ubicaciones más cercanas a una propiedad a partir de sus coordenadas geográficas. **NO** es necesario que realicen la implementación. Si tuvieras que entrevistar a un experto inmobiliario para mapear barrios entre datasets, ¿qué 3 preguntas le harías para validar esa correspondencia?
  5. Si las coordenadas geoespaciales estuvieran disponibles, como las usarian?

Pueden leer otras columnas del conjunto de AirBnB además de las que están en `interesting_cols`, si les parecen relevantes.

¿Qué cosas no están en los datos que te gustaría tener para predecir mejor el precio de una propiedad?


### Criterios de evaluación
Se evaluará principalmente:
- claridad del código,
- justificación de las decisiones de curación,
- coherencia entre el análisis realizado y las conclusiones,
- presencia de validaciones después de operaciones críticas como merges o cargas a base.

No se espera una única solución correcta, pero sí que las decisiones estén justificadas y sean consistentes con los datos.


## Ejercicio 3:

Crear y guardar un nuevo conjunto de datos con todas las transformaciones realizadas anteriormente.

## Ejercicios opcionales:

El notebook `02.2 ETLs-DAGs.ipynb` tiene un esqueleto de referencia para guiarse.

1. Armar un script en python (archivo .py) [ETL](https://towardsdatascience.com/what-to-log-from-python-etl-pipelines-9e0cfe29950e) que corra los pasos de extraccion, transformacion y carga, armando una funcion para cada etapa del proceso y luego un main que corra todos los pasos requeridos.

2. Armar un DAG en Apache Airflow que corra el ETL. (https://airflow.apache.org/docs/apache-airflow/stable/tutorial.html)

3. Bonus: embeddings y búsqueda semántica con descripciones de AirBnB.
   - Usar `sentence-transformers` para codificar descripciones textuales de propiedades.
   - Tomar un subconjunto chico de descripciones, calcular embeddings y encontrar el par más similar con similitud coseno.
   - Reflexionar: ¿por qué este resultado no se puede lograr con `LIKE '%keyword%'` en SQL? ¿Qué pasa si dos propiedades son similares pero usan palabras distintas? ¿Qué representan los 384 números del embedding?

4. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?


5. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?

Ejemplo conceptual:


In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Valores únicos con posibles inconsistencias
council_values = melb_df['CouncilArea'].dropna().unique().tolist()

message = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": f"""Estos son los valores únicos de la columna CouncilArea en un dataset de propiedades de Melbourne:
{council_values}

Identificá: (1) duplicados con distinta capitalización o spelling,
 (2) valores que parecen errores, (3) valores que podrían agruparse.
Respondé en JSON con la estructura: {{"estandarizado": {{"valor_original": "valor_correcto"}}}}"""
    }]
)

mapping = json.loads(message.content[0].text)
melb_df['CouncilArea_clean'] = melb_df['CouncilArea'].map(
    mapping.get('estandarizado', {})
).fillna(melb_df['CouncilArea'])
